# Client Folder Structure Creator

Don't copy this into the client's data folder. You should just run this from wherever it lives on your computer. It will create all the relevant files and notebooks and folders for the client 

Configure the variables in the first code cell and run the notebook.

Options:
- period: 'baseline' | 'pilot' 
- data_type: 'procurement' | 'serving' | 'both'
- data_format: 'pdf' | 'tabular' | 'both'

Pro tip! This is configured so that if you have already done the baseline analysis and you use this script to generate pilot files, then it will actually copy and paste the baseline analysis files over. So all you have to do is tweak them. 

# Setup

In [ ]:
import os
import shutil
from pathlib import Path

from dotenv import load_dotenv

# Load .env from parent directory
env_path = Path("../.env")
if not env_path.exists():
    raise FileNotFoundError(
        f"❌ .env file not found at {env_path.resolve()}\n   Please create a .env "
        f"file with CLIENT_DATA_FOLDER set to your client data directory."
    )

load_dotenv(env_path)

In [ ]:
# === CLIENT INFORMATION ===
client_name = ""  # <-- set to actual client name (string)
sub_client = None  # e.g ["Vendor_1", "Vendor_2"] or None

# === PERIOD AND DATA SETTINGS ===
period = "baseline"  # 'baseline' | 'pilot'
data_type = "serving"  # 'procurement' | 'serving' | 'both'
data_format = "tabular"  # 'pdf' | 'tabular' | 'both'

# === OPTIONAL FEATURES ===
include_initial_data_check = False  # Include exploratory notebook (0 initial_data_check.ipynb)
dry_run = False  # Set to True to preview without creating files/folders

# === PATHS ===
# Get from .env - must be set in .env file
client_data_folder = os.getenv("CLIENT_DATA_FOLDER")
# client_data_folder = ""

if not client_data_folder:
    raise ValueError(
        f"❌ CLIENT_DATA_FOLDER not found in .env file at {env_path.resolve()}\n   "
        f"Please add: CLIENT_DATA_FOLDER=/path/to/your/client/data/folder"
    )

In [ ]:
# Validation sets
VALID_PERIODS = {"baseline", "pilot"}
VALID_DATA_TYPES = {"procurement", "purchasing", "POS", "point of sale", "sales", "serving", "both"}
VALID_DATA_FORMATS = {"pdf", "tabular", "both"}

DATA_DIR_MAP = {
    "procurement": ["purchasing-procurement data"],
    "purchasing": ["purchasing-procurement data"],
    "serving": ["sales-serving data"],
    "POS": ["sales-serving data"],
    "point of sale": ["sales-serving data"],
    "sales": ["sales-serving data"],
    "both": ["purchasing-procurement data", "sales-serving data"],
}

# Validate configuration


def validate_config():
    """Validate all configuration parameters."""
    errors = []

    if not client_name.strip():
        errors.append("client_name cannot be empty")
    if client_data_folder is None:
        errors.append("client_data_folder must be set")
    if period not in VALID_PERIODS:
        errors.append(f"period must be one of {sorted(VALID_PERIODS)}")
    if data_type not in VALID_DATA_TYPES:
        errors.append(f"data_type must be one of {sorted(VALID_DATA_TYPES)}")
    if data_format not in VALID_DATA_FORMATS:
        errors.append(f"data_format must be one of {sorted(VALID_DATA_FORMATS)}")
    if sub_client is not None and not (
        isinstance(sub_client, list)
        and all(isinstance(sc, str) and sc.strip() for sc in sub_client)
    ):
        errors.append("sub_client must be None or a list of non-empty strings")

    if errors:
        raise ValueError("❌ Configuration errors:\n  • " + "\n  • ".join(errors))


validate_config()

# Expand configuration
client_data_folder_expanded = Path(client_data_folder).expanduser()
template_notebook_folder = Path(".")
client_dir = client_data_folder_expanded / client_name
data_formats = ["pdf", "tabular"] if data_format == "both" else [data_format]
sub_clients = sub_client if sub_client is not None else [None]

# Validate paths exist
if not template_notebook_folder.exists():
    raise FileNotFoundError(
        f"❌ Template notebook folder not found: {template_notebook_folder.resolve()}"
    )

print("✅ Configuration validated successfully!")
# Track created vs existing items
created_dirs = []
existing_dirs = []
created_notebooks = []
existing_notebooks = []

if dry_run:
    print("🔍 DRY RUN MODE - No files or folders will be created\n")

print(f"Client folder will be created at: {client_dir}")
print(f"Looking for template notebooks in: {template_notebook_folder.resolve()}")
if sub_client:
    print(f"Sub-clients: {sub_client}")
print()

# Functions

In [ ]:
def ensure_dir(path):
    """Create directory if it doesn't exist, track the result."""
    if path.exists():
        existing_dirs.append(path)
    else:
        if not dry_run:
            path.mkdir(parents=True)
        created_dirs.append(path)


def should_copy_notebook(notebook_name, period, formats, include_initial_check):
    """Determine if a notebook should be copied based on period and data format."""
    # Skip initial data check unless explicitly requested
    if notebook_name == "0 initial_data_check.ipynb" and not include_initial_check:
        return False

    # Skip pilot analysis for baseline period
    if period == "baseline" and notebook_name == "5 pilot_analysis.ipynb":
        return False

    # Skip PDF extraction if only tabular format requested
    if "pdf" not in formats and notebook_name == "1a extract_pdf_data.ipynb":
        return False

    # Skip tabular extraction if only PDF format requested
    tabular_notebooks = {
        "1b extract_tabular_data.ipynb",
        "0.5. Prepare tabular data Runscript.py",
    }
    if "tabular" not in formats and notebook_name in tabular_notebooks:  # noqa: SIM103
        return False

    return True


def copy_notebooks(source_dir, dest_dir, should_filter=True):
    """Copy template files from source to destination with optional filtering."""
    if not source_dir.exists():
        return

    notebook_candidates = list(source_dir.glob("*.ipynb"))
    notebook_candidates.extend(
        [
            source_dir / "0.5. Prepare tabular data Runscript.py",
            source_dir / "1.5. Clean Units Runscript.py",
        ]
    )

    for notebook in notebook_candidates:
        if not notebook.exists():
            continue
        if "setup_folder_structure.ipynb" in notebook.name:
            continue

        if should_filter and not should_copy_notebook(
            notebook.name, period, data_formats, include_initial_data_check
        ):
            continue

        dest = dest_dir / notebook.name
        if dest.exists():
            existing_notebooks.append(dest)
        else:
            if not dry_run:
                shutil.copy2(notebook, dest)
            created_notebooks.append(dest)

# Make the folders

In [ ]:
# Create directory structure
ensure_dir(client_dir)

period_root = client_dir / period
ensure_dir(period_root)

for data_folder in DATA_DIR_MAP[data_type]:
    data_root = period_root / data_folder
    ensure_dir(data_root)

    for sub_client_name in sub_clients:
        # If sub_client is specified, create sub-client folder level
        if sub_client_name is not None:
            current_data_root = data_root / sub_client_name
            ensure_dir(current_data_root)
        else:
            current_data_root = data_root

        raw_data_dir = current_data_root / "raw_data"
        ensure_dir(raw_data_dir)

        # Copy template notebooks to each sub-client folder
        # For pilot period, copy from baseline folder if it exists
        if period == "pilot":
            baseline_data_root = client_dir / "baseline" / data_folder
            if sub_client_name is not None:
                baseline_data_root = baseline_data_root / sub_client_name

            if baseline_data_root.exists():
                # Copy notebooks from baseline (without filtering)
                copy_notebooks(baseline_data_root, current_data_root, should_filter=False)
            else:
                print(f"⚠️  Warning: No baseline data found at {baseline_data_root}")
                print("   Only pilot analysis template will be copied.\n")

            # Also copy pilot analysis template
            pilot_analysis_template = template_notebook_folder / "5 pilot_analysis.ipynb"
            if pilot_analysis_template.exists():
                dest = current_data_root / "5 pilot_analysis.ipynb"
                if dest.exists():
                    existing_notebooks.append(dest)
                else:
                    if not dry_run:
                        shutil.copy2(pilot_analysis_template, dest)
                    created_notebooks.append(dest)

        # For baseline period, copy from template folder
        else:
            copy_notebooks(template_notebook_folder, current_data_root, should_filter=True)

# Display results
print("=" * 50)
print("DIRECTORY CREATION RESULTS")
print("=" * 50)
for path in created_dirs:
    print(f"✓ Created: {path}")
for path in existing_dirs:
    print(f"• Already exists: {path}")

if created_notebooks or existing_notebooks:
    print("\n" + "=" * 50)
    print("NOTEBOOK COPY RESULTS")
    print("=" * 50)
    for path in created_notebooks:
        print(f"✓ Copied: {path}")
    for path in existing_notebooks:
        print(f"• Already exists: {path}")

# Summary statistics
print("\n" + "=" * 50)
print("📊 SUMMARY")
print("=" * 50)
print(f"   Directories created: {len(created_dirs)}")
print(f"   Directories already existed: {len(existing_dirs)}")
print(f"   Notebooks copied: {len(created_notebooks)}")
print(f"   Notebooks already existed: {len(existing_notebooks)}")

if dry_run:
    print("\n🔍 DRY RUN - No actual changes were made")
else:
    print("\n✅ Setup complete!")

In [ ]:
def print_directory_tree(root: Path, max_depth: int = 3, prefix: str = ""):
    """Print a simple directory tree structure including files."""
    if max_depth < 0:
        return

    entries = sorted(root.iterdir(), key=lambda p: (not p.is_dir(), p.name.lower()))
    for i, entry in enumerate(entries):
        is_last = i == len(entries) - 1
        connector = "└── " if is_last else "├── "
        print(f"{prefix}{connector}{entry.name}")
        if entry.is_dir():
            next_prefix = prefix + ("    " if is_last else "│   ")
            print_directory_tree(entry, max_depth - 1, next_prefix)


print(f"\n=== Directory Structure for {client_name} ===")
print(client_dir.name)
print_directory_tree(client_dir, max_depth=3)